In [2]:
import ollama 

In [3]:

inventory_db = {
        "laptop": {"stock": 5, "base_price": 1200},
        "monitor": {"stock": 0, "base_price": 300},
        "keyboard": {"stock": 25, "base_price": 80}
    }

In [4]:
#DFINING TOOL 
# TOOL 1:CHECKING DATABASE
def check_inventory(product_name):
    product_name = product_name.lower()
    
    if product_name in inventory_db:
        return inventory_db[product_name]
    
    return {"stock": 0, "base_price": None}





# TOOL 2: Business Logic for Discounts
def calculate_loyalty_discount(base_price, years_as_customer):
    discount = min(years_as_customer * 0.05, 0.30)  # max 30%
    final_price = base_price * (1 - discount)
    return round(final_price, 2)

In [5]:

# 1. Map the functions so the script can call them by name
available_functions = {
    'check_inventory': check_inventory,
    'calculate_loyalty_discount': calculate_loyalty_discount,
}

In [6]:
#DEFINING SCHEMA
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Get stock and price for a product",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_loyalty_discount",
            "description": "Calculate final price based on loyalty years",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "number"},
                    "years_as_customer": {"type": "integer"}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [7]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customer': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [9]:

#“I am a customer for 5 years. What will be the final price of a laptop?"
#"I want to buy an iPhone. Can you check stock?"
message=[
        {"role": "user", "content":"I am a customer for 5 years. What will be the final price of a laptop?"}
    ]

#CALLING llM
response = ollama.chat(
    model="qwen3-vl:4b",
    messages=message
   ,
    tools=tools
)

print(response)

model='qwen3-vl:4b' created_at='2026-10-01T14:09:08.9504491Z' done=True done_reason='stop' total_duration=36648973600 load_duration=10773946900 prompt_eval_count=248 prompt_eval_duration=150173000 eval_count=1255 eval_duration=25703758000 message=Message(role='assistant', content='', thinking='Okay, let me try to figure out how to approach this. The user is asking about the final price of a laptop after considering their 5 years as a customer. They mentioned they\'re a customer for 5 years, so I need to use the loyalty discount function.\n\nWait, but the problem is, the function calculate_loyalty_discount requires base_price and years_as_customer. The user hasn\'t provided the base price yet. Hmm, so I need to check the inventory first to get the stock and price for the laptop. But wait, the user didn\'t specify the product name. Oh, the function check_inventory requires product_name. The user said "laptop," so maybe the product name is "laptop." \n\nSo first, I should call check_inven

In [10]:
response["message"]

Message(role='assistant', content='', thinking='Okay, let me try to figure out how to approach this. The user is asking about the final price of a laptop after considering their 5 years as a customer. They mentioned they\'re a customer for 5 years, so I need to use the loyalty discount function.\n\nWait, but the problem is, the function calculate_loyalty_discount requires base_price and years_as_customer. The user hasn\'t provided the base price yet. Hmm, so I need to check the inventory first to get the stock and price for the laptop. But wait, the user didn\'t specify the product name. Oh, the function check_inventory requires product_name. The user said "laptop," so maybe the product name is "laptop." \n\nSo first, I should call check_inventory with product_name "laptop" to get the base price. Then, once I have the base price, use calculate_loyalty_discount with base_price and years_as_customer 5. \n\nWait, but the user is asking for the final price, so I need both steps. But maybe 

In [11]:
# Check if the model asked to use any tools
tool_calls = response['message'].get('tool_calls')

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']

        # Get the actual Python function
        function_to_call = available_functions[tool_name]

        # Run the function with given arguments
        result = function_to_call(**tool_args)

        # Add the model's tool request to the conversation
        message.append(response['message'])

        # Add the tool's result back to the conversation
        message.append({
            'role': 'tool',
            'content': str(result)
        })

message


[{'role': 'user',
  'content': 'I am a customer for 5 years. What will be the final price of a laptop?'},
 Message(role='assistant', content='', thinking='Okay, let me try to figure out how to approach this. The user is asking about the final price of a laptop after considering their 5 years as a customer. They mentioned they\'re a customer for 5 years, so I need to use the loyalty discount function.\n\nWait, but the problem is, the function calculate_loyalty_discount requires base_price and years_as_customer. The user hasn\'t provided the base price yet. Hmm, so I need to check the inventory first to get the stock and price for the laptop. But wait, the user didn\'t specify the product name. Oh, the function check_inventory requires product_name. The user said "laptop," so maybe the product name is "laptop." \n\nSo first, I should call check_inventory with product_name "laptop" to get the base price. Then, once I have the base price, use calculate_loyalty_discount with base_price and 

In [12]:
# Ask the model to generate the final answer
final_response = ollama.chat(
    model='qwen3-vl:4b',
    messages=message
)

print(final_response['message']['content'])

{"name": "calculate_loyalty_discount", "arguments": {"base_price": 1200, "years_as_customer": 5}}
{"name": "calculate_loyalty_discount", "arguments": {"base_price": 1200, "years_as_customer": 5}}
